# **DSK821: Natural Language Processing**

In this tutorial we will use text representation in two small NLP applications:

1. News clustering with DBSCAN
2. IMDb sentiment classification


In [1]:
import numpy as np
import pandas as pd
import spacy

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import DBSCAN
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# nlp = spacy.load("en_core_web_md")


# 1. **Application 1**: **News Clustering with DBSCAN**

We now apply the same idea to real text.

Pipeline:

**News headlines → TF-IDF vectors → cosine distance → DBSCAN clusters**

DBSCAN is useful here because it does not require us to specify the number of clusters in advance and can mark isolated headlines as **noise**.


In [2]:
news = pd.read_csv("english_news_headlines.csv")
news.head()


,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [3]:
# Use a small sample so the example runs quickly in Colab
headlines = news['Title'].tolist()[:10000]
news_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X_news = news_vectorizer.fit_transform(headlines)

print("Number of headlines:", len(headlines))
print("TF-IDF matrix shape:", X_news.shape)


Number of headlines: 10000
TF-IDF matrix shape: (10000, 5000)


In [4]:
dbscan = DBSCAN(
    eps=0.5,
    min_samples=3,
    metric="cosine"
)

cluster_labels = dbscan.fit_predict(X_news)

print("Clusters found:", len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0))
print("Noise headlines:", np.sum(cluster_labels == -1))


Clusters found: 566
Noise headlines: 5352


### Inspect some clusters

`-1` is DBSCAN's label for **noise**. We display only actual clusters below.


In [5]:
cluster_df = pd.DataFrame({
    "headline": headlines,
    "cluster": cluster_labels
})

cluster_sizes = (
    cluster_df[cluster_df["cluster"] != -1]
    ["cluster"]
    .value_counts()
)

for cluster_id in cluster_sizes.head(5).index:
    print(f"\n--- Cluster {cluster_id} ({cluster_sizes[cluster_id]} headlines) ---")
    for headline in cluster_df.loc[
        cluster_df["cluster"] == cluster_id, "headline"
    ].head(8):
        print("•", headline)



--- Cluster 0 (1214 headlines) ---
• Oil and Economy Cloud Stocks' Outlook (Reuters)
• Safety Net (Forbes.com)
• Oil and Economy Cloud Stocks' Outlook
• Google IPO Auction Off to Rocky Start
• Google auction begins on Friday
• Steady as they go
• Google IPO: Type in 'confusing,' 'secrecy'
• Oldsmobile: The final parking lot

--- Cluster 226 (132 headlines) ---
• Invasive Purple Weed May Meet Its Match (AP)
• US ; Japanese-born teen youngest US chess champ since Bobby &lt;b&gt;...&lt;/b&gt;
• VARSITY MATCH
• Clinton: Ex-President Inaugurates China-Backed Internet Search &lt;b&gt;...&lt;/b&gt;
• High court hears dispute over Michigan interstate wine sales
• Sharon seeks support for planned coalition; Gaza violence kills &lt;b&gt;...&lt;/b&gt;
• Nortel announces Sprint extension worth US\$1B; Lucent claims US &lt;b&gt;...&lt;/b&gt;
• US Supreme Court hears dispute over interstate wine sales; Maine &lt;b&gt;...&lt;/b&gt;

--- Cluster 2 (113 headlines) ---
• Mauritian textile firm cuts job

### Try it yourself

Change `eps` or `min_samples` and observe what happens.

- Smaller `eps` → stricter similarity requirement
- Larger `eps` → more headlines may join the same cluster
- Larger `min_samples` → stronger requirement for forming a cluster


# 2. **Application 2: IMDb Sentiment Classification**

Finally, we use TF-IDF vectors as features for a supervised classifier.

Pipeline:

**Movie reviews → TF-IDF → Logistic Regression → positive / negative**

The lecture version uses a stratified sample of **5,000 IMDb reviews**.


In [6]:
imdb = pd.read_csv("imdb_5000.csv")

imdb.head()

,review,sentiment
0,I got the DVD very cheap and I'm a total Drewb...,negative
1,I liked this comedy so much. Will Smith does n...,positive
2,What a piece of junk this movie was. The premi...,negative
3,"I am a big fan of the movie, but not for the u...",positive
4,Im going to keep this fairly brief as to not s...,positive


In [7]:
print("Dataset size:", len(imdb))
print("\nClass distribution:")
print(imdb["sentiment"].value_counts())
print("\nProportions:")
print(imdb["sentiment"].value_counts(normalize=True).round(3))


Dataset size: 5000

Class distribution:
sentiment
negative    2500
positive    2500
Name: count, dtype: int64

Proportions:
sentiment
negative    0.5
positive    0.5
Name: proportion, dtype: float64


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    imdb["review"],
    imdb["sentiment"],
    test_size=0.20,
    random_state=42,
    stratify=imdb["sentiment"]
)

print("Training reviews:", len(X_train))
print("Test reviews:", len(X_test))


Training reviews: 4000
Test reviews: 1000


## 7.1 Convert the reviews to TF-IDF vectors

The vectorizer is **fit only on the training data**.  
The test data is transformed using the vocabulary learned from the training set.


In [9]:
sentiment_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = sentiment_vectorizer.fit_transform(X_train)
X_test_tfidf = sentiment_vectorizer.transform(X_test)

print("Training matrix:", X_train_tfidf.shape)
print("Test matrix:", X_test_tfidf.shape)


Training matrix: (4000, 5000)
Test matrix: (1000, 5000)


## 7.2 Train a Logistic Regression classifier


In [10]:
classifier = LogisticRegression(max_iter=500)
classifier.fit(X_train_tfidf, y_train)

y_pred = classifier.predict(X_test_tfidf)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print()
print(classification_report(y_test, y_pred))


Accuracy: 0.839

              precision    recall  f1-score   support

    negative       0.86      0.81      0.83       500
    positive       0.82      0.87      0.84       500

    accuracy                           0.84      1000
   macro avg       0.84      0.84      0.84      1000
weighted avg       0.84      0.84      0.84      1000



In [11]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf = RandomForestClassifier( max_depth=10, n_estimators=300)
rf.fit(X_train_tfidf, y_train)

y_pred = rf.predict(X_test_tfidf)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.805

              precision    recall  f1-score   support

    negative       0.84      0.75      0.79       500
    positive       0.77      0.86      0.82       500

    accuracy                           0.81      1000
   macro avg       0.81      0.80      0.80      1000
weighted avg       0.81      0.81      0.80      1000



In [12]:
import numpy as np
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, classification_report

tf.keras.utils.set_random_seed(42)

# Convert sparse TF-IDF matrices to dense arrays
X_train_nn = X_train_tfidf.toarray().astype("float32")
X_test_nn = X_test_tfidf.toarray().astype("float32")

# Convert sentiment labels into binary values
# positive = 1, negative = 0
y_train_nn = (y_train == "positive").astype("float32").to_numpy()
y_test_nn = (y_test == "positive").astype("float32").to_numpy()

print("Training data:", X_train_nn.shape)
print("Test data:", X_test_nn.shape)

Training data: (4000, 5000)
Test data: (1000, 5000)


/Users/fangsiyu/dev-env/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train_nn.shape[1],)),

    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
    
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │       320,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 322,689 (1.23 MB)

 Trainable params: 322,689 (1.23 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
history = model.fit(
    X_train_nn,
    y_train_nn,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7466 - loss: 0.5929 - val_accuracy: 0.8487 - val_loss: 0.3807
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9331 - loss: 0.1937 - val_accuracy: 0.8350 - val_loss: 0.3902
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9894 - loss: 0.0510 - val_accuracy: 0.8388 - val_loss: 0.4829
Epoch 4/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9981 - loss: 0.0167 - val_accuracy: 0.8338 - val_loss: 0.5532
Epoch 5/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9994 - loss: 0.0062 - val_accuracy: 0.8425 - val_loss: 0.5889
Epoch 6/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9997 - loss: 0.0019 - val_accuracy: 0.8400 - val_loss: 0.6237
Epoch 7/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 1.0000 - loss: 9.2136e-04 - val_accuracy: 0.8413 - val_loss: 0.6519
Epoch 8/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 1.0000 - loss: 6.1526e-04 - val_accu

In [17]:
# Predict probabilities
y_prob = model.predict(X_test_nn).ravel()

# Convert probabilities to class labels
y_pred_nn = (y_prob >= 0.5).astype(int)

print("Neural Network Accuracy:",
      round(accuracy_score(y_test_nn, y_pred_nn), 3))

print()
print(classification_report(
    y_test_nn,
    y_pred_nn,
    target_names=["negative", "positive"]
))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 453us/step
Neural Network Accuracy: 0.826

              precision    recall  f1-score   support

    negative       0.82      0.83      0.83       500
    positive       0.83      0.82      0.83       500

    accuracy                           0.83      1000
   macro avg       0.83      0.83      0.83      1000
weighted avg       0.83      0.83      0.83      1000



## Try your own reviews

The important point is the complete NLP pipeline:

**raw text → vector representation → trained model → prediction**


In [24]:
examples = [
    # "This movie was fantastic! I really loved it.",
    "I hate it.",
    "not bad actually",
    "The plot was boring and the acting was terrible."
]

examples_tfidf = sentiment_vectorizer.transform(examples)
predictions = classifier.predict(examples_tfidf)

for review, prediction in zip(examples, predictions):
    print(f"{prediction:>8}  |  {review}")


positive  |  I hate it.
negative  |  not bad actually
negative  |  The plot was boring and the acting was terrible.
